In [3]:
!pip install scikit-learn

import sys
sys.path.append('./Property-awareness-Representation-Learning')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from models import PropertyPredictor
from utils import load_mat, CombinedDataset, loss_function


In [4]:
# Load the .mat files
shape_data = load_mat("ShapeSpace.mat")["ShapeSpace"]        # (50, 50, 248396)
property_data = load_mat("PropertySpace.mat")["PropertySpace"]  # (5, 248396)

# Fix shape: (50, 50, 248396) → (248396, 50, 50)
shape_data = np.transpose(shape_data, (1, 2, 0))
shape_data = shape_data.reshape(248396, 50, 50)
property_data = property_data.T  # (248396, 5)

# Convert to float and wrap into a dataset
dataset = CombinedDataset(shape_data.astype(np.float32), property_data.astype(np.float32))

# Train-test split
train_dataset, test_dataset = train_test_split(dataset, test_size=0.1, random_state=42)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, drop_last=True)


In [5]:
input_dim = 50 * 50       # 2500
hidden_dim = 64
output_dim = 5

model = PropertyPredictor(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [6]:
epochs = 10  # You can change this

model.train()
for epoch in range(epochs):
    total_loss = 0
    for x, y in train_loader:
        x = x.view(x.size(0), -1).to(device)  # Flatten: [32, 50, 50] → [32, 2500]
        y = y.to(device)
    
        optimizer.zero_grad()
        y_pred = model(x)
        loss = loss_function(y_pred, y)
        loss.backward()
        optimizer.step()
    
        total_loss += loss.item()


    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


Epoch 1/10, Loss: 5.2269
Epoch 2/10, Loss: 5.0125
Epoch 3/10, Loss: 4.9785
Epoch 4/10, Loss: 4.9712
Epoch 5/10, Loss: 4.9604
Epoch 6/10, Loss: 4.9536
Epoch 7/10, Loss: 4.9405
Epoch 8/10, Loss: 4.9330
Epoch 9/10, Loss: 4.9222
Epoch 10/10, Loss: 4.9122


In [7]:
# Testing the MLP Prediction Model
model.eval()  # Switch to evaluation mode (turns off dropout, etc.)
test_loss = 0

with torch.no_grad():  # Don't calculate gradients during testing
    for x, y in test_loader:
        x = x.view(x.size(0), -1).to(device)  # Flatten: [32, 50, 50] → [32, 2500]
        y = y.to(device)

        y_pred = model(x)
        loss = loss_function(y_pred, y)
        test_loss += loss.item()

avg_test_loss = test_loss / len(test_loader)
print(f"\n Average test loss: {avg_test_loss:.4f}")



 Average test loss: 5.0362


In [8]:
import torch
torch.save(model.state_dict(), "mlp_model.pt")
